Github Analysis!

In [1]:
from spark_utils import quick_start
spark = quick_start("TestConnection")

🚀 Creating Spark session: TestConnection
📡 Connecting to: spark://spark-master:7077
🗄️  MinIO endpoint: http://minio:9000
🔺 Delta Lake: ENABLED


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/24 05:02:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/24 05:02:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark 4.0.0 session created successfully!
🔗 Spark UI: http://localhost:4041
💡 S3A ready for s3a://delta-lake/ operations
🔺 Delta Lake ready for delta table operations!


In [2]:
# Load your data
df = spark.read.format("delta").load("s3a://delta-lake/gold/gold_github_technology_adoption")


25/06/24 05:03:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [3]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- technology: string (nullable = true)
 |-- analysis_date: string (nullable = true)
 |-- daily_mentions: long (nullable = true)
 |-- daily_repo_mentions: long (nullable = true)
 |-- daily_event_mentions: long (nullable = true)
 |-- primary_repository: string (nullable = true)
 |-- adoption_status: string (nullable = true)
 |-- technology_category: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- activity_level: string (nullable = true)
 |-- is_trending: boolean (nullable = true)
 |-- investment_signal: string (nullable = true)
 |-- recommendation: string (nullable = true)
 |-- data_source: string (nullable = true)
 |-- confidence_level: string (nullable = true)
 |-- processing_pipeline: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)



In [4]:
df.show(5, truncate=False)

25/06/24 05:03:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+--------------+-------------+--------------+-------------------+--------------------+--------------------------------+---------------+-------------------+--------------+--------------+-----------+-----------------+--------------+------------+----------------+-------------------+--------------------------+
|date      |technology    |analysis_date|daily_mentions|daily_repo_mentions|daily_event_mentions|primary_repository              |adoption_status|technology_category|market_segment|activity_level|is_trending|investment_signal|recommendation|data_source |confidence_level|processing_pipeline|processed_at              |
+----------+--------------+-------------+--------------+-------------------+--------------------+--------------------------------+---------------+-------------------+--------------+--------------+-----------+-----------------+--------------+------------+----------------+-------------------+--------------------------+
|2025-01-02|accelerate    |2025-01-02   |10

In [5]:
df.groupBy("technology") \
  .sum("daily_mentions") \
  .orderBy("sum(daily_mentions)", ascending=False) \
  .show(10, truncate=False)

+--------------+-------------------+
|technology    |sum(daily_mentions)|
+--------------+-------------------+
|ethereum      |123686             |
|near          |121942             |
|docker        |73136              |
|npm           |52114              |
|python        |51407              |
|c             |41852              |
|go            |40497              |
|react         |38168              |
|github actions|37299              |
|azure         |33852              |
+--------------+-------------------+
only showing top 10 rows


25/06/24 05:24:34 WARN TransportChannelHandler: Exception in connection from /172.18.0.12:34394
java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.SocketChannelImpl.throwConnectionReset(SocketChannelImpl.java:394)
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:426)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:255)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:356)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteChannel.java:151)
	at io.netty.channel.nio.NioEventLoop.processSelectedKey(NioEventLoop.java:796)
	at io.netty.channel.nio.NioEventLoop.processSelectedKeysOptimized(NioEventLoop.java:732)
	at io.netty.channel.nio.NioEventLoop.processSelectedKeys(NioEventLoop.java:658)
	at io.netty.channel.nio.NioEventLoop.run(NioEventLoop.java:562)
	at io.netty.util.concurrent.SingleThreadEventEx

In [7]:
df.filter(df.technology != "github") \
  .select("technology", "data_quality_score", "source_record_count") \
  .orderBy("data_quality_score", ascending=False) \
  .show(20, truncate=False)

+--------------+------------------+-------------------+
|technology    |data_quality_score|source_record_count|
+--------------+------------------+-------------------+
|ethereum      |53.46236337253335 |24                 |
|near          |53.41977533713698 |24                 |
|docker        |51.55621949961254 |24                 |
|python        |51.37972063126884 |24                 |
|nuget         |51.32966871338033 |24                 |
|typescript    |51.15459676374836 |24                 |
|npm           |51.07567718269169 |24                 |
|git           |51.0372601404527  |24                 |
|github actions|51.03056459880534 |24                 |
|c             |50.9722804575799  |24                 |
|azure         |50.89467803225715 |24                 |
|go            |50.85329738961686 |24                 |
|react         |50.814441295466565|24                 |
|php           |50.68579908545487 |24                 |
|java          |50.68415264078749 |24           

In [8]:
df.filter((df.technology != "github") & (df.data_quality_score > 50)) \
  .orderBy(df.daily_mentions.desc()) \
  .select("technology", "daily_mentions", "data_quality_score", "primary_repository") \
  .show(20, truncate=False)

+--------------+--------------+------------------+----------------------------------------+
|technology    |daily_mentions|data_quality_score|primary_repository                      |
+--------------+--------------+------------------+----------------------------------------+
|ethereum      |31544         |53.46236337253335 |perspectivefi/token-list                |
|near          |31156         |53.41977533713698 |Xelp66/Lava                             |
|docker        |14178         |51.55621949961254 |QYG2297248353/appstore-1panel           |
|python        |12570         |51.37972063126884 |signalfyi/support                       |
|nuget         |12114         |51.32966871338033 |soenneker/soenneker.runners.nuget       |
|typescript    |10519         |51.15459676374836 |bendemboski/fractal-page-object         |
|npm           |9800          |51.07567718269169 |GingertronMk1/clubhouse-laravel         |
|git           |9450          |51.0372601404527  |yasuhirokimura/freebsd-ports  

25/06/23 16:36:17 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.8: Command exited with code 137
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_18_34!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_14_7!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_18_27!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_14_36!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_14_26!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_18_25!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_14_12!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_14_10!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_18_7!
25/06/23 16:36:17 WARN BlockManagerMasterEndpoint: No more rep

In [9]:
df.filter((df.daily_mentions > 5000) & (df.data_quality_score > 51)) \
  .orderBy(df.daily_mentions.desc()) \
  .select("technology", "daily_mentions", "data_quality_score", "primary_repository") \
  .show(truncate=False)

+--------------+--------------+------------------+---------------------------------+
|technology    |daily_mentions|data_quality_score|primary_repository               |
+--------------+--------------+------------------+---------------------------------+
|github        |455527        |100.0             |regro/cf-graph-countyfair        |
|ethereum      |31544         |53.46236337253335 |perspectivefi/token-list         |
|near          |31156         |53.41977533713698 |Xelp66/Lava                      |
|docker        |14178         |51.55621949961254 |QYG2297248353/appstore-1panel    |
|python        |12570         |51.37972063126884 |signalfyi/support                |
|nuget         |12114         |51.32966871338033 |soenneker/soenneker.runners.nuget|
|typescript    |10519         |51.15459676374836 |bendemboski/fractal-page-object  |
|npm           |9800          |51.07567718269169 |GingertronMk1/clubhouse-laravel  |
|git           |9450          |51.0372601404527  |yasuhirokimura/

In [5]:
# Load your data
df = spark.read.format("delta").load("s3a://delta-lake/gold/gold_github_technology_adoption")

In [8]:
df.show(15, truncate=False)

+----------+-------------------+-------------+--------------+-------------------+--------------------+-------------------------------------------------+---------------+-------------------+--------------+--------------+-----------+-----------------+--------------+------------+----------------+-------------------+--------------------------+
|date      |technology         |analysis_date|daily_mentions|daily_repo_mentions|daily_event_mentions|primary_repository                               |adoption_status|technology_category|market_segment|activity_level|is_trending|investment_signal|recommendation|data_source |confidence_level|processing_pipeline|processed_at              |
+----------+-------------------+-------------+--------------+-------------------+--------------------+-------------------------------------------------+---------------+-------------------+--------------+--------------+-----------+-----------------+--------------+------------+----------------+-------------------+-----

25/06/23 18:45:17 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.11: Command exited with code 137
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_7!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_49_32!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_49!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_45!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_49_8!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_14!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_39!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_31!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_49_33!
25/06/23 18:45:17 WARN BlockManagerMasterEndpoint: No more re